# RAG-Augmented Fine-tuning

Fine-tunes BERT, HateBERT, and RoBERTa from **base (non-fine-tuned) weights**
with retrieval-augmented inputs.

For each model, the pipeline:
1. Loads the model's own base encoder as the retriever
2. Retrieves k nearest neighbors from `index/{model}/base/vdb_training.faiss`, self-excluded at train time
3. Builds an augmented input: `query [SEP] [hate] neighbor1 [SEP] [not hate] neighbor2 ...`
4. Fine-tunes a classifier head on the augmented inputs
5. Trains on both IHC and ISHate datasets

**Outputs:**
```
weights_rag/{model}/base/{dataset}/
```

## 1. Imports

In [ ]:
import os
import json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import faiss
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from datasets import load_dataset, Dataset
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')
from rag import encode, retrieve_top_k_above_threshold

## 2. Configuration

In [ ]:
RAG_DIR         = Path('.')
WEIGHTS_DIR     = Path('..') / 'weights'
WEIGHTS_RAG_DIR = Path('..') / 'weights_rag'
INDEX_DIR       = RAG_DIR / 'index'

MODELS = {
    'bert':     'bert-base-uncased',
    'hatebert': 'GroNLP/hateBERT',
    'roberta':  'roberta-base',
}

# Retrieval config
K = 5
THRESHOLD = 0.98

# Training config
MAX_LENGTH    = 256
BATCH_SIZE    = 16
LEARNING_RATE = 2e-5
NUM_EPOCHS    = 3


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
print(f'k      : {K}')

## 3. Load Datasets

Same IHC and ISHate splits as `baseline.ipynb`.

In [ ]:
raw_ihc = load_dataset('tasksource/implicit-hate-stg1', split='train')
splits  = raw_ihc.train_test_split(test_size=0.10, seed=42)

def add_binary_label_ihc(example):
    example['label'] = 0 if example['class'] == 'not_hate' else 1
    return example

train_ihc = splits['train'].map(add_binary_label_ihc)
test_ihc  = splits['test'].map(add_binary_label_ihc)

ishate_raw = load_dataset('BenjaminOcampo/ISHate')

def add_binary_label_ishate(example):
    example['label'] = 0 if example['hateful_layer'] == 'Non-HS' else 1
    return example

train_ishate = ishate_raw['train'].map(add_binary_label_ishate)
test_ishate  = ishate_raw['test'].map(add_binary_label_ishate)

DATASETS = {
    'IHC':    {'train': train_ihc,    'test': test_ihc,    'text_col': 'post'},
    'ISHate': {'train': train_ishate, 'test': test_ishate, 'text_col': 'text'},
}

print(f'IHC    — train: {len(train_ihc):,}  test: {len(test_ihc):,}')
print(f'ISHate — train: {len(train_ishate):,}  test: {len(test_ishate):,}')

## 4. Self-Exclusion Lookup

`chunks_training.csv` maps raw tweet text → `chunk_id` in the FAISS index.
Used at train time to pass `chunk_id` so a model never retrieves itself as a neighbor.

In [ ]:
chunks_df = pd.read_csv(RAG_DIR / 'chunks' / 'chunks_training.csv')

def strip_label_prefix(text):
    return text.replace('[hate] ', '', 1).replace('[not hate] ', '', 1)

text_to_chunk_id = {
    strip_label_prefix(row.text): int(row.chunk_id)
    for _, row in chunks_df.iterrows()
}
print(f'Self-exclusion lookup: {len(text_to_chunk_id):,} entries')

## 5. Augmentation Function

Retrieves k neighbors for a dataset split using an explicit retriever (model/tokenizer/index).
Called once per model inside the training loop — retrieval changes across models.

**Input format:** `{query} {sep} {[hate] neighbor1} {sep} {[not hate] neighbor2} ...`

- Query: **no** label prefix (that is what the model must predict).
- Neighbors: **keep** their `[hate]`/`[not hate]` prefix as few-shot context clues.

In [ ]:
def augment_split(hf_dataset, text_col, is_train, ret_model, ret_tokenizer, ret_index, ret_documents):
    records = []
    for example in tqdm(hf_dataset, desc=f"{'train' if is_train else 'test'}"):
        tweet    = example[text_col]
        chunk_id = text_to_chunk_id.get(tweet) if is_train else None
        neighbors = retrieve_top_k_above_threshold(tweet, THRESHOLD, ret_model, ret_tokenizer, ret_index, ret_documents,
                                   chunk_id=chunk_id, k=K)
        records.append({
            'query':     tweet,
            'neighbors': [text for text, _ in neighbors],
            'label':     example['label'],
        })
    return records

## 6. Tokenization

Assembles the augmented string using the model's `sep_token` then tokenizes.

In [ ]:
def tokenize_augmented(records, tokenizer, max_length=MAX_LENGTH):
    sep = tokenizer.sep_token
    texts = [
        f' {sep} '.join([r['query']] + r['neighbors'])
        for r in records
    ]
    labels = [r['label'] for r in records]
    encoded = tokenizer(
        texts,
        truncation=True,
        padding='max_length',
        max_length=max_length,
    )
    encoded['labels'] = labels
    return Dataset.from_dict(encoded)

## 7. Metrics

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'macro_f1': f1_score(labels, preds, average='macro',  zero_division=0),
        'macro_p':  precision_score(labels, preds, average='macro', zero_division=0),
        'macro_r':  recall_score(labels, preds, average='macro',    zero_division=0),
    }

## 8. Training Loop

For each model (bert, hatebert, roberta):
1. Load the model's own base encoder as retriever
2. Augment both IHC and ISHate datasets with that retriever
3. Free the retriever, then train a classifier for each dataset
4. Save weights and print classification report

In [ ]:
results = {}

with open(INDEX_DIR / 'lookup_training.json') as f:
    ret_documents = json.load(f)

for model_name, hf_id in MODELS.items():
    print(f"\n{'#'*60}")
    print(f'# Retriever + classifier: {model_name} / base')
    print(f"{'#'*60}")

    # ── 1. Load base retriever ──────────────────────────────────────
    ret_tokenizer = AutoTokenizer.from_pretrained(hf_id)
    ret_model     = AutoModel.from_pretrained(hf_id).eval().to(device)
    ret_index     = faiss.read_index(str(INDEX_DIR / model_name / 'base' / 'vdb_training.faiss'))
    print(f'Retrieval index: {ret_index.ntotal:,} vectors')

    # ── 2. Augment both datasets with this model's retriever ────────
    aug_data = {}
    for ds_name, ds_cfg in DATASETS.items():
        print(f'\n=== Augmenting {ds_name} ===')
        aug_data[ds_name] = {
            'train': augment_split(ds_cfg['train'], ds_cfg['text_col'], True,
                                   ret_model, ret_tokenizer, ret_index, ret_documents),
            'test':  augment_split(ds_cfg['test'],  ds_cfg['text_col'], False,
                                   ret_model, ret_tokenizer, ret_index, ret_documents),
        }

    # Free retriever and tokenizer memory before training
    del ret_model, ret_tokenizer
    if device.type == 'cuda':
        torch.cuda.empty_cache()

    results[model_name] = {}

    # ── 3. Train a classifier for each dataset ──────────────────────
    for ds_name in aug_data:
        print(f"\n{'='*60}")
        print(f'Model: {model_name}  |  Weights: base  |  Dataset: {ds_name}')
        print(f"{'='*60}")

        tokenizer = AutoTokenizer.from_pretrained(hf_id)
        tok_train = tokenize_augmented(aug_data[ds_name]['train'], tokenizer)
        tok_test  = tokenize_augmented(aug_data[ds_name]['test'],  tokenizer)

        model = AutoModelForSequenceClassification.from_pretrained(hf_id, num_labels=2)

        save_path = str(WEIGHTS_RAG_DIR / model_name / 'base' / ds_name)
        os.makedirs(save_path, exist_ok=True)

        training_args = TrainingArguments(
            output_dir=f'./checkpoints_rag/{model_name}/base/{ds_name}',
            num_train_epochs=NUM_EPOCHS,
            per_device_train_batch_size=BATCH_SIZE,
            per_device_eval_batch_size=BATCH_SIZE * 2,
            learning_rate=LEARNING_RATE,
            eval_strategy='epoch',
            save_strategy='no',
            logging_strategy='epoch',
            report_to='none',
            seed=42,
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=tok_train,
            eval_dataset=tok_test,
            compute_metrics=compute_metrics,
        )

        trainer.train()
        trainer.save_model(save_path)
        tokenizer.save_pretrained(save_path)
        print(f'  Weights saved -> {save_path}')

        preds_out = trainer.predict(tok_test)
        preds  = np.argmax(preds_out.predictions, axis=-1)
        labels = [r['label'] for r in aug_data[ds_name]['test']]

        print(classification_report(labels, preds, target_names=['Non-HS', 'HS']))

        results[model_name][ds_name] = {
            'macro_f1': f1_score(labels, preds, average='macro',  zero_division=0),
            'macro_p':  precision_score(labels, preds, average='macro', zero_division=0),
            'macro_r':  recall_score(labels, preds, average='macro',    zero_division=0),
        }

        del model
        if device.type == 'cuda':
            torch.cuda.empty_cache()

## 9. Results

One table per dataset. Rows = model (base retriever + base classifier).

In [ ]:
for ds_name in DATASETS:
    rows = {model_name: results[model_name][ds_name] for model_name in MODELS}
    df = pd.DataFrame(rows).T
    df.columns = ['Macro F1', 'Macro Precision', 'Macro Recall']
    df.index.name = 'Model'
    styled = (
        df.style
        .format('{:.3f}')
        .highlight_max(axis=0, props='font-weight: bold; background-color: #d4f1d4')
        .set_caption(f'RAG Fine-tuning (base retriever) — {ds_name}')
    )
    display(styled)